In [ ]:
import cv2
import os
import glob
import re
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display

In [ ]:
# === Paths ===
input_folder = r"E:\Fatemeh\20250609-PH\S5"
output_video = r"E:\Fatemeh\20250609-PH\S5_Overlay\overlay_ratio_timelapse.mp4"

In [ ]:
# === Parameters ===
frame_rate = 5
skip_frames = 1
line_y_position = 1000  # Scanline height
epsilon = 1e-3  # Prevent divide-by-zero in ratio

In [ ]:
# === Find all channel 0 files and sort by time index ===
image_c0_files = sorted(glob.glob(os.path.join(input_folder, "*c0*.tif")))
if not image_c0_files:
    print("❌ No c0 images found. Check naming.")
    raise SystemExit()

def extract_time_index(filename):
    match = re.search(r"t(\d+)", filename)
    return int(match.group(1)) if match else float('inf')

image_c0_files.sort(key=extract_time_index)

In [ ]:
# === Preview scanline on first c0 image ===
preview_img = cv2.imread(image_c0_files[0], cv2.IMREAD_GRAYSCALE)
if preview_img is None:
    print("❌ Could not load first c0 image for preview.")
else:
    img_with_line = cv2.cvtColor(preview_img, cv2.COLOR_GRAY2RGB)
    cv2.line(img_with_line, (0, line_y_position), (img_with_line.shape[1]-1, line_y_position), (255, 0, 0), 2)
    plt.figure(figsize=(10, 6))
    plt.imshow(img_with_line)
    plt.title(f"Preview: Scanline at y = {line_y_position}")
    plt.axis("on")
    plt.show()

In [ ]:
# === Video setup ===
video_writer = None
frame_count = 0

for i, c0_file in enumerate(image_c0_files):
    if i % skip_frames != 0:
        continue

    if i % 10 == 0:
        print(f"⏳ Processing frame {i}/{len(image_c0_files)}")

    # Derive c1 filename from c0
    c1_file = c0_file.replace("c0", "c1")
    if not os.path.exists(c1_file):
        print(f"⚠️ Missing c1 file for {c0_file}, skipping.")
        continue

    # === Load both channel images ===
    img_c0 = cv2.imread(c0_file, cv2.IMREAD_GRAYSCALE)
    img_c1 = cv2.imread(c1_file, cv2.IMREAD_GRAYSCALE)

    if img_c0 is None or img_c1 is None:
        print(f"⚠️ Skipping unreadable pair: {c0_file}")
        continue

    if line_y_position >= img_c0.shape[0]:
        print(f"⚠️ Scanline y={line_y_position} out of bounds.")
        continue

    # === Optional: enhance c0 image for visualization ===
    def adjust_gamma(image, gamma=1.2):
        inv_gamma = 1.0 / gamma
        table = np.array([(i / 255.0) ** inv_gamma * 255 for i in np.arange(256)]).astype("uint8")
        return cv2.LUT(image, table)

    img_c0_disp = adjust_gamma(img_c0, gamma=1.5)
    img_overlay = cv2.cvtColor(img_c0_disp, cv2.COLOR_GRAY2BGR)

    # === Extract scanline profiles ===
    profile_c0 = img_c0[line_y_position, :].astype(np.float32)
    profile_c1 = img_c1[line_y_position, :].astype(np.float32)

    # === Compute ratio profile ===
    ratio_profile = profile_c0 / (profile_c1 + epsilon)  # Avoid divide-by-zero

    # === Noise cancellation ===
    kernel_size = 21
    kernel = np.ones(kernel_size) / kernel_size

    smoothed_ratio_profile = np.convolve(ratio_profile, kernel, mode='valid')

    # === Normalize profile to visualize ===
    smoothed_ratio_profile = smoothed_ratio_profile - np.min(smoothed_ratio_profile)
    smoothed_ratio_profile = smoothed_ratio_profile / np.max(smoothed_ratio_profile + 1e-6)
    # ratio_profile = np.clip(ratio_profile, 0, 5)  # Clip for visualization
    smoothed_ratio_profile = smoothed_ratio_profile * 400  # Scale for drawing
    ratio_scaled = smoothed_ratio_profile
    
    # === Overlay profile on image ===
    for x in range(1, len(ratio_scaled)):
        y1 = int(line_y_position - ratio_scaled[x - 1]*2) # Scaled to fit in view: Each intensity value is scaled (divided by 4) to map 0–255 range to a ~60-pixel band.
        y2 = int(line_y_position - ratio_scaled[x]*2)
        pt1 = (x - 1, y1)
        pt2 = (x, y2)
        cv2.line(img_overlay, pt1, pt2, (0, 0, 255), 3)  # Red profile 

    # Draw the base scanline in light gray
    cv2.line(img_overlay, (0, line_y_position), (img_c0.shape[1] - 1, line_y_position), (180, 180, 180), 2)

    # === Initialize writer if needed ===
    if video_writer is None:
        h, w = img_overlay.shape[:2]
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        video_writer = cv2.VideoWriter(output_video, fourcc, frame_rate, (w, h))

    video_writer.write(img_overlay)
    frame_count += 1
    
# === Finalize ===
video_writer.release()
cv2.destroyAllWindows()
print(f"✅ Ratio profile video saved to: {output_video}")
print(f"🎞️ Total frames written: {frame_count}")

# === Show video (Jupyter only) ===
display(Video(output_video, embed=True))